# Solution 5C: Score-Tilted Market-Cap Weights
**BUSI 722: Data-Driven Finance II**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

preds = pd.read_parquet("expanding_predictions.parquet")
print(f"Predictions: {len(preds):,} stock-months, {preds['month'].min()} to {preds['month'].max()}")

preds["u"] = preds.groupby("month")["pred"].transform(lambda x: x.rank(pct=True))

Predictions: 66,121 stock-months, 2024-01 to 2025-11


## 1-3. Compare three weighting approaches

In [2]:
results = {}
for m, grp in preds.groupby("month"):
    u, ret, mcap = grp["u"].values, grp["return"].values, grp["marketcap"].values

    # Pure mcap
    w = mcap / mcap.sum()
    r_mcap = np.sum(w * ret)

    # Score-tilted
    w = mcap * (u ** 2)
    w = w / w.sum()
    r_tilt = np.sum(w * ret)
    n_tilt = (w > 0.01).sum()

    # Equal-weight top decile
    top = u >= np.percentile(u, 90)
    w = np.zeros(len(u))
    w[top] = 1.0 / top.sum() if top.sum() > 0 else 0
    r_eq = np.sum(w * ret)

    results[m] = {"Market-Cap": r_mcap, "Score-Tilted MCap": r_tilt,
                  "Equal-Wt Top Decile": r_eq,
                  "n_mcap": (mcap / mcap.sum() > 0.01).sum(),
                  "n_tilt": n_tilt, "n_eq": top.sum()}

tilt_df = pd.DataFrame(results).T.sort_index()

print(f"{'Approach':25s} {'Mean Mo':>10s} {'Ann Vol':>10s} {'Sharpe':>10s} {'Avg n>1%':>10s}")
print("-" * 67)
for col, nc in [("Market-Cap", "n_mcap"), ("Score-Tilted MCap", "n_tilt"),
                ("Equal-Wt Top Decile", "n_eq")]:
    s = tilt_df[col].astype(float)
    sr = s.mean() / s.std() * np.sqrt(12) if s.std() > 0 else 0
    print(f"{col:25s} {s.mean():10.4f} {s.std()*np.sqrt(12):10.4f} {sr:10.3f} {tilt_df[nc].mean():10.1f}")

Approach                     Mean Mo    Ann Vol     Sharpe   Avg n>1%
-------------------------------------------------------------------
Market-Cap                    0.0171     0.1155      1.773       12.5
Score-Tilted MCap             0.0144     0.1100      1.567       15.3
Equal-Wt Top Decile           0.0150     0.1922      0.935      287.9


## 4. Discussion

Score-tilted weights stay closer to the market-cap benchmark while overweighting stocks with higher predicted ranks. This avoids the extreme overweighting of micro-cap stocks that equal-weight sorts produce, reducing turnover and transaction costs while still capturing the alpha signal.